# SBET jump checker

Notebook pour :

- charger un SBET sur une fenêtre temporelle GPS `[t_start, t_end]`
- utiliser **ton loader SBET existant** si possible
- sinon tomber sur un **fallback reader** standard SBET binaire
- convertir la trajectoire en ENU local
- détecter des **sauts très précis** sur :
  - position
  - vitesse
  - altitude
  - attitude (`roll`, `pitch`, `heading`)
  - cohérence position ↔ vitesse
- lister et tracer les instants suspects

Le notebook est volontairement modulaire : tu peux juste modifier la cellule **CONFIG** et la cellule **IMPORT DU LOADER EXISTANT**.

In [ ]:
# ============================================================
# CONFIG
# ============================================================
from pathlib import Path

# SBET par défaut (à adapter si besoin)
sbet_path = Path("/media/b085164/Elements/CALIB_26_02_25/ODyN_calib/base_v0/in/reference.out")

# Fenêtre temporelle GPS (secondes GPS)
t_start = 305060.0
t_end   = 305760.0

# Affichage
max_display_points = 50000

# Si True, essaie d'abord d'utiliser ton loader existant
prefer_existing_loader = True

# Seuils de détection (tu peux les raffiner)
ROBUST_Z_THRESHOLD = 8.0
ABS_STEP_M_THRESHOLD = None      # ex: 0.50 pour forcer un seuil absolu en mètres
ABS_DALT_M_THRESHOLD = None      # ex: 0.20
ABS_DHEAD_RAD_THRESHOLD = None   # ex: np.deg2rad(1.0)

print("SBET:", sbet_path)
print(f"Time window: [{t_start}, {t_end}]")

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import importlib
import struct
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyproj import Transformer

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# ============================================================
# IMPORT DU LOADER EXISTANT
# ============================================================
# Essaie ici de brancher TON loader déjà existant.
# Le notebook teste plusieurs imports possibles, puis fallback automatiquement.
#
# Si tu connais déjà le bon import, remplace simplement cette cellule par ex. :
#
# from mon_module import load_sbet_existing
#
# def load_sbet_existing(path):
#     return load_sbet(path)
#
# Le loader doit idéalement retourner :
#   - soit un DataFrame avec une colonne temps + colonnes utiles
#   - soit un ndarray de shape (N, >= 10)

load_sbet_existing = None
existing_loader_name = None

candidate_imports = [
    # (module_name, function_name)
    ("navtools_PDM.sbet_tools", "load_sbet"),
    ("navtools_PDM.utils.sbet_tools", "load_sbet"),
    ("navtools_PDM.utils.trajectory", "load_sbet"),
    ("navtools_PDM.pipeline", "load_sbet"),
    ("sbet_tools", "load_sbet"),
    ("trajectory_tools", "load_sbet"),
]

if prefer_existing_loader:
    for mod_name, fn_name in candidate_imports:
        try:
            mod = importlib.import_module(mod_name)
            fn = getattr(mod, fn_name)
            load_sbet_existing = fn
            existing_loader_name = f"{mod_name}.{fn_name}"
            break
        except Exception:
            pass

if load_sbet_existing is not None:
    print(f"Existing loader found: {existing_loader_name}")
else:
    print("No existing loader auto-detected. Fallback binary reader will be used.")

In [ ]:
# ============================================================
# FALLBACK: standard binary SBET reader
# ============================================================
# Standard SBET record = 17 doubles, généralement :
# 0 time
# 1 lat
# 2 lon
# 3 alt
# 4 v_north
# 5 v_east
# 6 v_down
# 7 roll
# 8 pitch
# 9 heading
# 10 wander
# 11 a_x
# 12 a_y
# 13 a_z
# 14 ar_x
# 15 ar_y
# 16 ar_z
#
# NB: selon certaines chaînes, les conventions de vitesse/acc peuvent varier,
# mais cette lecture standard permet déjà l'analyse des discontinuités.

SBET_COLUMNS = [
    "time",
    "lat",
    "lon",
    "alt",
    "v_north",
    "v_east",
    "v_down",
    "roll",
    "pitch",
    "heading",
    "wander",
    "a_x",
    "a_y",
    "a_z",
    "ar_x",
    "ar_y",
    "ar_z",
]

def read_sbet_binary_standard(path):
    path = Path(path)
    raw = np.fromfile(path, dtype=np.float64)
    if raw.size % 17 != 0:
        raise ValueError(
            f"SBET size incompatible with 17-double records: {raw.size} values"
        )
    arr = raw.reshape(-1, 17)
    df = pd.DataFrame(arr, columns=SBET_COLUMNS)
    return df

In [ ]:
# ============================================================
# NORMALISATION DU FORMAT
# ============================================================
def standardize_sbet_dataframe(obj):
    """
    Convertit la sortie du loader existant OU du fallback en DataFrame standard.
    On essaie de reconnaître les colonnes usuelles.
    """
    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
    elif isinstance(obj, np.ndarray):
        if obj.ndim != 2 or obj.shape[1] < 10:
            raise ValueError(f"Unexpected ndarray shape: {obj.shape}")
        cols = SBET_COLUMNS[:obj.shape[1]]
        df = pd.DataFrame(obj, columns=cols)
    else:
        raise TypeError(f"Unsupported loader output type: {type(obj)}")

    rename_map = {}

    # Temps
    for c in df.columns:
        lc = str(c).lower()
        if lc in ["time", "gps_time", "gpstime", "sow"]:
            rename_map[c] = "time"

    # Latitude / longitude / altitude
    for c in df.columns:
        lc = str(c).lower()
        if lc in ["lat", "latitude"]:
            rename_map[c] = "lat"
        elif lc in ["lon", "longitude", "long"]:
            rename_map[c] = "lon"
        elif lc in ["alt", "altitude", "height", "h"]:
            rename_map[c] = "alt"

    # Vitesses
    for c in df.columns:
        lc = str(c).lower()
        if lc in ["v_north", "vn", "vel_n", "velnorth", "north_velocity"]:
            rename_map[c] = "v_north"
        elif lc in ["v_east", "ve", "vel_e", "veleast", "east_velocity"]:
            rename_map[c] = "v_east"
        elif lc in ["v_down", "vd", "vel_d", "veldn", "down_velocity"]:
            rename_map[c] = "v_down"

    # Angles
    for c in df.columns:
        lc = str(c).lower()
        if lc == "roll":
            rename_map[c] = "roll"
        elif lc == "pitch":
            rename_map[c] = "pitch"
        elif lc in ["heading", "yaw", "azimuth", "head"]:
            rename_map[c] = "heading"

    df = df.rename(columns=rename_map)

    required = ["time", "lat", "lon", "alt"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing essential SBET columns after normalization: {missing}")

    df = df.sort_values("time").reset_index(drop=True)
    return df

def load_sbet_any(path):
    if load_sbet_existing is not None:
        try:
            out = load_sbet_existing(path)
            df = standardize_sbet_dataframe(out)
            print(f"Loaded with existing loader: {existing_loader_name}")
            return df
        except Exception as e:
            warnings.warn(
                f"Existing loader failed ({existing_loader_name}): {e}\n"
                "Fallback binary reader will be used."
            )

    df = read_sbet_binary_standard(path)
    df = standardize_sbet_dataframe(df)
    print("Loaded with fallback standard SBET reader")
    return df

In [ ]:
# ============================================================
# CHARGEMENT + FENÊTRE TEMPORELLE
# ============================================================
df = load_sbet_any(sbet_path)

print("Raw columns:", list(df.columns))
print("Raw size:", len(df))

df = df[(df["time"] >= t_start) & (df["time"] <= t_end)].copy().reset_index(drop=True)

if len(df) < 3:
    raise ValueError(f"Not enough samples in requested time window: {len(df)}")

print("Windowed size:", len(df))
display(df.head())

In [ ]:
# ============================================================
# CONVERSION EN ENU LOCAL
# ============================================================
# On utilise le premier point comme origine locale.
# SBET standard: lat/lon en radians.
# Si ton loader existant renvoie déjà des degrés, une heuristique les détecte.

def maybe_rad_to_deg(series, name):
    max_abs = np.nanmax(np.abs(series.to_numpy()))
    # lat/lon en radians ~ <= pi ; en degrés ~ jusqu'à 180
    if name in ["lat", "lon"] and max_abs <= np.pi + 1e-3:
        return np.rad2deg(series.to_numpy()), "radians"
    return series.to_numpy(), "degrees"

lat_deg, lat_unit = maybe_rad_to_deg(df["lat"], "lat")
lon_deg, lon_unit = maybe_rad_to_deg(df["lon"], "lon")

print(f"Latitude interpreted as:  {lat_unit}")
print(f"Longitude interpreted as: {lon_unit}")

df["lat_deg"] = lat_deg
df["lon_deg"] = lon_deg

lat0 = float(df["lat_deg"].iloc[0])
lon0 = float(df["lon_deg"].iloc[0])
h0   = float(df["alt"].iloc[0])

# geodetic -> ECEF
to_ecef = Transformer.from_crs("EPSG:4979", "EPSG:4978", always_xy=True)

x, y, z = to_ecef.transform(
    df["lon_deg"].to_numpy(),
    df["lat_deg"].to_numpy(),
    df["alt"].to_numpy()
)

x0, y0, z0 = to_ecef.transform(lon0, lat0, h0)

# ECEF -> ENU local
lat0r = np.deg2rad(lat0)
lon0r = np.deg2rad(lon0)

R = np.array([
    [-np.sin(lon0r),               np.cos(lon0r),              0.0],
    [-np.sin(lat0r)*np.cos(lon0r), -np.sin(lat0r)*np.sin(lon0r), np.cos(lat0r)],
    [ np.cos(lat0r)*np.cos(lon0r),  np.cos(lat0r)*np.sin(lon0r), np.sin(lat0r)],
])

dxyz = np.vstack([x - x0, y - y0, z - z0])
enu = R @ dxyz

df["east"]  = enu[0]
df["north"] = enu[1]
df["up"]    = enu[2]

display(df[["time", "east", "north", "up"]].head())

In [ ]:
# ============================================================
# OUTILS NUMÉRIQUES
# ============================================================
def wrap_to_pi(a):
    return (a + np.pi) % (2 * np.pi) - np.pi

def robust_zscore(x):
    x = np.asarray(x, dtype=float)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    if not np.isfinite(mad) or mad < 1e-15:
        return np.full_like(x, np.nan, dtype=float)
    return 0.6744897501960817 * (x - med) / mad

def safe_diff(x):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x, dtype=float)
    out[:] = np.nan
    out[1:] = np.diff(x)
    return out

def safe_angle_diff(x):
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x, dtype=float)
    out[:] = np.nan
    out[1:] = wrap_to_pi(np.diff(x))
    return out

In [ ]:
# ============================================================
# MÉTRIQUES DE DISCONTINUITÉ
# ============================================================
t = df["time"].to_numpy()

dt = safe_diff(t)
df["dt"] = dt

# Pas position ENU
de = safe_diff(df["east"])
dn = safe_diff(df["north"])
du = safe_diff(df["up"])

df["step_3d_m"] = np.sqrt(de**2 + dn**2 + du**2)
df["step_h_m"]  = np.sqrt(de**2 + dn**2)
df["dalt_m"]    = du

# Vitesse issue des positions
df["speed_pos_3d"] = df["step_3d_m"] / df["dt"]
df["speed_pos_h"]  = df["step_h_m"] / df["dt"]

# Vitesse SBET si présente
has_vel = all(c in df.columns for c in ["v_north", "v_east", "v_down"])
if has_vel:
    df["speed_sbet_3d"] = np.sqrt(
        df["v_north"]**2 + df["v_east"]**2 + df["v_down"]**2
    )
    df["speed_sbet_h"] = np.sqrt(
        df["v_north"]**2 + df["v_east"]**2
    )
    df["speed_consistency_3d"] = np.abs(df["speed_pos_3d"] - df["speed_sbet_3d"])
    df["speed_consistency_h"]  = np.abs(df["speed_pos_h"] - df["speed_sbet_h"])
else:
    df["speed_sbet_3d"] = np.nan
    df["speed_sbet_h"] = np.nan
    df["speed_consistency_3d"] = np.nan
    df["speed_consistency_h"] = np.nan

# Accélération dérivée
if has_vel:
    dvn = safe_diff(df["v_north"])
    dve = safe_diff(df["v_east"])
    dvd = safe_diff(df["v_down"])
    df["acc_from_vel"] = np.sqrt(dvn**2 + dve**2 + dvd**2) / df["dt"]
else:
    sp = df["speed_pos_3d"].to_numpy()
    df["acc_from_vel"] = np.abs(safe_diff(sp) / df["dt"])

# Angles
has_angles = all(c in df.columns for c in ["roll", "pitch", "heading"])
if has_angles:
    df["droll"]  = safe_angle_diff(df["roll"])
    df["dpitch"] = safe_angle_diff(df["pitch"])
    df["dhead"]  = safe_angle_diff(df["heading"])

    df["roll_rate_num"]  = np.abs(df["droll"]  / df["dt"])
    df["pitch_rate_num"] = np.abs(df["dpitch"] / df["dt"])
    df["head_rate_num"]  = np.abs(df["dhead"]  / df["dt"])
else:
    for c in ["droll", "dpitch", "dhead", "roll_rate_num", "pitch_rate_num", "head_rate_num"]:
        df[c] = np.nan

display(df.head())

In [ ]:
# ============================================================
# SCORES ROBUSTES
# ============================================================
metric_cols = [
    "step_3d_m",
    "step_h_m",
    "dalt_m",
    "speed_pos_3d",
    "speed_consistency_3d",
    "acc_from_vel",
    "roll_rate_num",
    "pitch_rate_num",
    "head_rate_num",
]

for c in metric_cols:
    if c in df.columns:
        df[f"rz_{c}"] = np.abs(robust_zscore(df[c].to_numpy()))

score_cols = [c for c in df.columns if c.startswith("rz_")]
df["jump_score"] = np.nanmax(df[score_cols].to_numpy(), axis=1)

# Règles d'anomalie
flag = np.zeros(len(df), dtype=bool)

flag |= np.nan_to_num(df["jump_score"].to_numpy(), nan=0.0) > ROBUST_Z_THRESHOLD

if ABS_STEP_M_THRESHOLD is not None:
    flag |= np.nan_to_num(df["step_3d_m"].to_numpy(), nan=0.0) > ABS_STEP_M_THRESHOLD

if ABS_DALT_M_THRESHOLD is not None:
    flag |= np.abs(np.nan_to_num(df["dalt_m"].to_numpy(), nan=0.0)) > ABS_DALT_M_THRESHOLD

if ABS_DHEAD_RAD_THRESHOLD is not None:
    flag |= np.abs(np.nan_to_num(df["dhead"].to_numpy(), nan=0.0)) > ABS_DHEAD_RAD_THRESHOLD

df["is_jump_candidate"] = flag

print("Number of candidate jump samples:", int(df["is_jump_candidate"].sum()))

In [ ]:
# ============================================================
# TABLEAU DES CANDIDATS
# ============================================================
candidate_cols = [
    "time",
    "dt",
    "east",
    "north",
    "up",
    "step_3d_m",
    "step_h_m",
    "dalt_m",
    "speed_pos_3d",
    "speed_sbet_3d",
    "speed_consistency_3d",
    "acc_from_vel",
    "roll_rate_num",
    "pitch_rate_num",
    "head_rate_num",
    "jump_score",
]

cand = (
    df.loc[df["is_jump_candidate"], [c for c in candidate_cols if c in df.columns]]
      .sort_values("jump_score", ascending=False)
      .reset_index(drop=True)
)

display(cand.head(100))

In [ ]:
# ============================================================
# REGROUPEMENT EN ÉVÉNEMENTS
# ============================================================
# Plusieurs échantillons suspects consécutifs sont regroupés en un seul événement.

def group_consecutive_times(times, max_gap_s=0.25):
    if len(times) == 0:
        return []
    times = np.asarray(times, dtype=float)
    groups = [[times[0]]]
    for ti in times[1:]:
        if ti - groups[-1][-1] <= max_gap_s:
            groups[-1].append(ti)
        else:
            groups.append([ti])
    return groups

jump_times = df.loc[df["is_jump_candidate"], "time"].to_numpy()
groups = group_consecutive_times(jump_times, max_gap_s=max(0.25, 3*np.nanmedian(df["dt"])))

events = []
for g in groups:
    t0, t1 = g[0], g[-1]
    sub = df[(df["time"] >= t0) & (df["time"] <= t1)]
    i_peak = sub["jump_score"].idxmax()
    row = df.loc[i_peak]
    events.append({
        "t_start": t0,
        "t_end": t1,
        "duration_s": t1 - t0,
        "t_peak": row["time"],
        "peak_score": row["jump_score"],
        "peak_step_3d_m": row.get("step_3d_m", np.nan),
        "peak_dalt_m": row.get("dalt_m", np.nan),
        "peak_speed_consistency_3d": row.get("speed_consistency_3d", np.nan),
        "peak_head_rate_num": row.get("head_rate_num", np.nan),
    })

events_df = pd.DataFrame(events).sort_values("peak_score", ascending=False).reset_index(drop=True)
display(events_df.head(50))

In [ ]:
# ============================================================
# PLOTS
# ============================================================
def decimate_df(dfi, max_points=max_display_points):
    if len(dfi) <= max_points:
        return dfi
    idx = np.linspace(0, len(dfi) - 1, max_points).astype(int)
    return dfi.iloc[idx].copy()

plot_df = decimate_df(df)

jump_plot = plot_df["is_jump_candidate"].to_numpy()
tplot = plot_df["time"].to_numpy()

# 1) Trajectoire locale
plt.figure(figsize=(9, 8))
plt.plot(plot_df["east"], plot_df["north"], lw=0.8, label="trajectory")
plt.scatter(
    plot_df.loc[plot_df["is_jump_candidate"], "east"],
    plot_df.loc[plot_df["is_jump_candidate"], "north"],
    s=18,
    label="jump candidates"
)
plt.xlabel("East [m]")
plt.ylabel("North [m]")
plt.title("SBET trajectory in local ENU")
plt.axis("equal")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

# 2) Step distance
plt.figure(figsize=(14, 4))
plt.plot(tplot, plot_df["step_3d_m"], lw=0.8)
plt.scatter(
    plot_df.loc[plot_df["is_jump_candidate"], "time"],
    plot_df.loc[plot_df["is_jump_candidate"], "step_3d_m"],
    s=16
)
plt.xlabel("GPS time [s]")
plt.ylabel("3D step [m]")
plt.title("Inter-sample 3D step")
plt.grid(True, alpha=0.3)
plt.show()

# 3) Speed consistency
if "speed_consistency_3d" in plot_df.columns:
    plt.figure(figsize=(14, 4))
    plt.plot(tplot, plot_df["speed_consistency_3d"], lw=0.8)
    plt.scatter(
        plot_df.loc[plot_df["is_jump_candidate"], "time"],
        plot_df.loc[plot_df["is_jump_candidate"], "speed_consistency_3d"],
        s=16
    )
    plt.xlabel("GPS time [s]")
    plt.ylabel("|speed_pos - speed_sbet| [m/s]")
    plt.title("Position / SBET speed consistency")
    plt.grid(True, alpha=0.3)
    plt.show()

# 4) Attitude numerical rates
if "head_rate_num" in plot_df.columns:
    plt.figure(figsize=(14, 4))
    plt.plot(tplot, plot_df["roll_rate_num"],  lw=0.8, label="roll")
    plt.plot(tplot, plot_df["pitch_rate_num"], lw=0.8, label="pitch")
    plt.plot(tplot, plot_df["head_rate_num"],  lw=0.8, label="heading")
    plt.scatter(
        plot_df.loc[plot_df["is_jump_candidate"], "time"],
        plot_df.loc[plot_df["is_jump_candidate"], "head_rate_num"],
        s=16,
        label="jump candidates"
    )
    plt.xlabel("GPS time [s]")
    plt.ylabel("Angular rate [rad/s]")
    plt.title("Numerical attitude rates")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

# 5) Global jump score
plt.figure(figsize=(14, 4))
plt.plot(tplot, plot_df["jump_score"], lw=0.8)
plt.scatter(
    plot_df.loc[plot_df["is_jump_candidate"], "time"],
    plot_df.loc[plot_df["is_jump_candidate"], "jump_score"],
    s=16
)
plt.axhline(ROBUST_Z_THRESHOLD, ls="--", lw=1.0)
plt.xlabel("GPS time [s]")
plt.ylabel("Robust jump score")
plt.title("Combined anomaly score")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# DIAGNOSTIC RAPIDE
# ============================================================
print("Time span loaded:", df['time'].min(), "->", df['time'].max())
print("Median dt [s]:", np.nanmedian(df["dt"]))
print("95th percentile step_3d [m]:", np.nanpercentile(df["step_3d_m"], 95))
print("99th percentile step_3d [m]:", np.nanpercentile(df["step_3d_m"], 99))

if "speed_consistency_3d" in df.columns and df["speed_consistency_3d"].notna().any():
    print("95th percentile |speed_pos - speed_sbet| [m/s]:", np.nanpercentile(df["speed_consistency_3d"], 95))
    print("99th percentile |speed_pos - speed_sbet| [m/s]:", np.nanpercentile(df["speed_consistency_3d"], 99))

if "head_rate_num" in df.columns and df["head_rate_num"].notna().any():
    print("99th percentile heading rate [deg/s]:", np.rad2deg(np.nanpercentile(df["head_rate_num"], 99)))

In [ ]:
# ============================================================
# EXPORT OPTIONNEL DES CANDIDATS / ÉVÉNEMENTS
# ============================================================
export_dir = sbet_path.parent / "jump_check_exports"
export_dir.mkdir(exist_ok=True, parents=True)

cand_path = export_dir / f"{sbet_path.stem}_jump_candidates_{int(t_start)}_{int(t_end)}.csv"
events_path = export_dir / f"{sbet_path.stem}_jump_events_{int(t_start)}_{int(t_end)}.csv"

cand.to_csv(cand_path, index=False)
events_df.to_csv(events_path, index=False)

print("Saved:")
print(" -", cand_path)
print(" -", events_path)

## Notes pratiques

Les métriques les plus utiles pour détecter un vrai saut sont souvent :

1. **`step_3d_m`** : saut spatial échantillon → échantillon  
2. **`speed_consistency_3d`** : incohérence entre la vitesse déduite des positions et la vitesse SBET  
3. **`head_rate_num` / `roll_rate_num` / `pitch_rate_num`** : micro-sauts d’attitude  
4. **`jump_score`** : score robuste combiné  

Pour aller encore plus loin ensuite, on pourra ajouter :

- comparaison orientation → variation de trajectoire
- projection du mouvement dans le repère corps
- détection de changement de covariance locale sur fenêtres glissantes
- score multi-échelle (instantané + fenêtre 1 s + fenêtre 5 s)
- comparaison de deux SBET sur la même fenêtre